In [37]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix, accuracy_score, recall_score, f1_score,
    roc_auc_score, balanced_accuracy_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

In [38]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [39]:
df = pd.read_csv('/kaggle/input/chronic-kidney-disease-dataset-analysis/Chronic_Kidney_Dsease_data.csv')
df.head()

,PatientID,Age,Gender,Ethnicity,SocioeconomicStatus,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,...,Itching,QualityOfLifeScore,HeavyMetalsExposure,OccupationalExposureChemicals,WaterQuality,MedicalCheckupsFrequency,MedicationAdherence,HealthLiteracy,Diagnosis,DoctorInCharge
0,1,71,0,0,0,2,31.069414,1,5.128112,1.676220,...,7.556302,76.076800,0,0,1,1.018824,4.966808,9.871449,1,Confidential
1,2,34,0,0,1,3,29.692119,1,18.609552,8.377574,...,6.836766,40.128498,0,0,0,3.923538,8.189275,7.161765,1,Confidential
2,3,80,1,1,0,1,37.394822,1,11.882429,9.607401,...,2.144722,92.872842,0,1,1,1.429906,7.624028,7.354632,1,Confidential
3,4,40,0,2,0,1,31.329680,0,16.020165,0.408871,...,7.077188,90.080321,0,0,0,3.226416,3.282688,6.629587,1,Confidential
4,5,43,0,1,1,2,23.726311,0,7.944146,0.780319,...,3.553118,5.258372,0,0,1,0.285466,3.849498,1.437385,1,Confidential


In [40]:
df = df.drop(columns=["PatientID", "DoctorInCharge"], errors="ignore")
df.head()

,Age,Gender,Ethnicity,SocioeconomicStatus,EducationLevel,BMI,Smoking,AlcoholConsumption,PhysicalActivity,DietQuality,...,MuscleCramps,Itching,QualityOfLifeScore,HeavyMetalsExposure,OccupationalExposureChemicals,WaterQuality,MedicalCheckupsFrequency,MedicationAdherence,HealthLiteracy,Diagnosis
0,71,0,0,0,2,31.069414,1,5.128112,1.676220,0.240386,...,4.518513,7.556302,76.076800,0,0,1,1.018824,4.966808,9.871449,1
1,34,0,0,1,3,29.692119,1,18.609552,8.377574,6.503233,...,2.202222,6.836766,40.128498,0,0,0,3.923538,8.189275,7.161765,1
2,80,1,1,0,1,37.394822,1,11.882429,9.607401,2.104828,...,5.967271,2.144722,92.872842,0,1,1,1.429906,7.624028,7.354632,1
3,40,0,2,0,1,31.329680,0,16.020165,0.408871,6.964422,...,2.176387,7.077188,90.080321,0,0,0,3.226416,3.282688,6.629587,1
4,43,0,1,1,2,23.726311,0,7.944146,0.780319,3.097796,...,6.800993,3.553118,5.258372,0,0,1,0.285466,3.849498,1.437385,1


In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1659 entries, 0 to 1658
Data columns (total 52 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Age                            1659 non-null   int64  
 1   Gender                         1659 non-null   int64  
 2   Ethnicity                      1659 non-null   int64  
 3   SocioeconomicStatus            1659 non-null   int64  
 4   EducationLevel                 1659 non-null   int64  
 5   BMI                            1659 non-null   float64
 6   Smoking                        1659 non-null   int64  
 7   AlcoholConsumption             1659 non-null   float64
 8   PhysicalActivity               1659 non-null   float64
 9   DietQuality                    1659 non-null   float64
 10  SleepQuality                   1659 non-null   float64
 11  FamilyHistoryKidneyDisease     1659 non-null   int64  
 12  FamilyHistoryHypertension      1659 non-null   i

In [42]:
X = df.drop("Diagnosis", axis=1)
y = df["Diagnosis"]

In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print("Train bincount:", np.bincount(y_train))
print("Test  bincount:", np.bincount(y_test))

Train bincount: [ 108 1219]
Test  bincount: [ 27 305]


In [44]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [45]:
# smote = SMOTE(random_state=42, k_neighbors=5)
# X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

In [46]:
# print("Before SMOTE:", np.bincount(y_train))
# print("After  SMOTE:", np.bincount(y_train_smote))

In [47]:
# classes = np.unique(y_train)
# weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
# class_weights = {int(classes[0]): float(weights[0]), int(classes[1]): float(weights[1])}
# print("Class weights:", class_weights)

class_weights = {0: 2.0, 1: 1.0}
print("Class weights:", class_weights)

Class weights: {0: 2.0, 1: 1.0}


In [48]:
from tensorflow import keras
from tensorflow.keras import layers, regularizers

def build_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.30),

        layers.Dense(64, activation="relu", kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.25),

        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=5e-4),
        loss="binary_crossentropy",
        metrics=[keras.metrics.BinaryAccuracy(name="acc"), keras.metrics.AUC(name="auc")]
    )
    return model

In [49]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True
)

In [50]:
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.20,
    epochs=400,
    batch_size=32,
    callbacks=[early_stop],
    class_weight=class_weights,
    verbose=1
)

Epoch 1/400
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - acc: 0.9189 - auc: 0.8864 - loss: 0.3074 - val_acc: 0.9135 - val_auc: 0.7202 - val_loss: 0.2960
Epoch 2/400
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - acc: 0.9401 - auc: 0.9163 - loss: 0.2679 - val_acc: 0.9023 - val_auc: 0.7332 - val_loss: 0.2988
Epoch 3/400
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - acc: 0.9413 - auc: 0.9399 - loss: 0.2519 - val_acc: 0.8947 - val_auc: 0.7313 - val_loss: 0.3021
Epoch 4/400
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - acc: 0.9333 - auc: 0.9331 - loss: 0.2579 - val_acc: 0.8985 - val_auc: 0.7324 - val_loss: 0.3047
Epoch 5/400
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - acc: 0.9443 - auc: 0.9456 - loss: 0.2427 - val_acc: 0.9023 - val_auc: 0.7373 - val_loss: 0.3083
Epoch 6/400
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - acc: 0.9491 - auc: 0.9341 - loss: 0.2453 - val_acc: 0.8985 - val_auc: 0.7376 - val_loss: 0.3136
Epoch 7/400
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - acc: 0.9412 - auc: 0.9484 - loss: 0.2293 - val_acc: 0

In [51]:
y_prob = model.predict(X_test_scaled).ravel()

print("\nProbability stats:")
print("Min:", float(y_prob.min()), "Max:", float(y_prob.max()), "Mean:", float(y_prob.mean()))
neg_probs = y_prob[y_test.values == 0]
pos_probs = y_prob[y_test.values == 1]
print("Neg probs -> Min:", float(neg_probs.min()), "Max:", float(neg_probs.max()), "Mean:", float(neg_probs.mean()))
print("Pos probs -> Min:", float(pos_probs.min()), "Max:", float(pos_probs.max()), "Mean:", float(pos_probs.mean()))

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 

Probability stats:
Min: 0.2838023006916046 Max: 0.9996787905693054 Mean: 0.8921363353729248
Neg probs -> Min: 0.44282811880111694 Max: 0.9990411400794983 Mean: 0.7645570039749146
Pos probs -> Min: 0.2838023006916046 Max: 0.9996787905693054 Mean: 0.9034302830696106


In [52]:
best_t, best_ba = 0.5, -1.0
for t in np.arange(0.05, 0.96, 0.01):
    y_pred_t = (y_prob >= t).astype(int)
    ba = balanced_accuracy_score(y_test, y_pred_t)
    if ba > best_ba:
        best_ba, best_t = ba, t

print("\nBest threshold (Balanced Accuracy):", round(best_t, 2), "Best BA:", round(best_ba, 4))

# Final predictions with best threshold
y_pred = (y_prob >= best_t).astype(int)


Best threshold (Balanced Accuracy): 0.79 Best BA: 0.6871


In [53]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

accuracy    = accuracy_score(y_test, y_pred)
sensitivity = tp / (tp + fn) if (tp + fn) else 0.0   # recall
recall      = recall_score(y_test, y_pred)
specificity = tn / (tn + fp) if (tn + fp) else 0.0
f1          = f1_score(y_test, y_pred)

roc_auc = roc_auc_score(y_test, y_prob)

print("\nConfusion Matrix:")
print("TN FP")
print(tn, fp)
print("FN TP")
print(fn, tp)

print(f"\nAccuracy     : {accuracy:.4f}")
print(f"Sensitivity  : {sensitivity:.4f}")
print(f"Recall       : {recall:.4f}")
print(f"Specificity  : {specificity:.4f}")
print(f"F1 Score     : {f1:.4f}")
print(f"ROC-AUC      : {roc_auc:.4f}")
print(f"Balanced Acc : {balanced_accuracy_score(y_test, y_pred):.4f}")


Confusion Matrix:
TN FP
14 13
FN TP
44 261

Accuracy     : 0.8283
Sensitivity  : 0.8557
Recall       : 0.8557
Specificity  : 0.5185
F1 Score     : 0.9016
ROC-AUC      : 0.7349
Balanced Acc : 0.6871


# Optuna

In [57]:
import numpy as np
import pandas as pd
import optuna

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers


classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = {int(classes[0]): float(weights[0]), int(classes[1]): float(weights[1])}
print("Class weights:", class_weights)


def build_ann(trial, input_dim: int):
    n_layers = trial.suggest_int("n_layers", 1, 4)
    first_units = trial.suggest_int("units_0", 32, 256, step=32)

    l2_val = trial.suggest_float("l2", 1e-6, 1e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    lr = trial.suggest_float("lr", 1e-5, 5e-3, log=True)

    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    # First layer
    model.add(layers.Dense(
        first_units,
        activation="relu",
        kernel_regularizer=regularizers.l2(l2_val)
    ))
    if trial.suggest_categorical("use_bn_0", [True, False]):
        model.add(layers.BatchNormalization())
    model.add(layers.Dropout(dropout))

    # Additional hidden layers
    units_prev = first_units
    for i in range(1, n_layers):
        units_i = trial.suggest_int(f"units_{i}", 16, units_prev, step=16)
        model.add(layers.Dense(
            units_i,
            activation="relu",
            kernel_regularizer=regularizers.l2(l2_val)
        ))
        if trial.suggest_categorical(f"use_bn_{i}", [True, False]):
            model.add(layers.BatchNormalization())
        model.add(layers.Dropout(dropout))
        units_prev = units_i

    # Output
    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc")]
    )
    return model


def objective(trial):
    tf.keras.backend.clear_session()

    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
    epochs = 400

    model = build_ann(trial, input_dim=X_train_scaled.shape[1])

    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True
    )

    model.fit(
        X_train_scaled, y_train,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        class_weight=class_weights,
        verbose=0
    )


    y_prob = model.predict(X_test_scaled, verbose=0).ravel()

    best_ba = -1
    for t in np.arange(0.05, 0.96, 0.01):
        y_pred = (y_prob >= t).astype(int)
        ba = balanced_accuracy_score(y_test, y_pred)
        if ba > best_ba:
            best_ba = ba

    return best_ba


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)  # increase to 50/100 if you want

print("\nBest Balanced Accuracy:", study.best_value)
print("Best Params:\n", study.best_params)


best_params = study.best_params
tf.keras.backend.clear_session()

def build_from_best(input_dim):
    # Rebuild model using best params (same logic as build_ann)
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    l2_val = best_params["l2"]
    dropout = best_params["dropout"]

    n_layers = best_params["n_layers"]
    units0 = best_params["units_0"]

    model.add(layers.Dense(units0, activation="relu", kernel_regularizer=regularizers.l2(l2_val)))
    if best_params["use_bn_0"]:
        model.add(layers.BatchNormalization())
    model.add(layers.Dropout(dropout))

    units_prev = units0
    for i in range(1, n_layers):
        units_i = best_params[f"units_{i}"]
        model.add(layers.Dense(units_i, activation="relu", kernel_regularizer=regularizers.l2(l2_val)))
        if best_params[f"use_bn_{i}"]:
            model.add(layers.BatchNormalization())
        model.add(layers.Dropout(dropout))
        units_prev = units_i

    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=best_params["lr"]),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc")]
    )
    return model

final_model = build_from_best(X_train_scaled.shape[1])

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=25,
    restore_best_weights=True
)

final_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=500,
    batch_size=best_params["batch_size"],
    callbacks=[early_stop],
    class_weight=class_weights,
    verbose=1
)


y_prob = final_model.predict(X_test_scaled, verbose=0).ravel()

# best threshold by balanced accuracy
best_t, best_ba = 0.5, -1
for t in np.arange(0.05, 0.96, 0.01):
    y_pred = (y_prob >= t).astype(int)
    ba = balanced_accuracy_score(y_test, y_pred)
    if ba > best_ba:
        best_ba, best_t = ba, t

y_pred = (y_prob >= best_t).astype(int)

from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, recall_score
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

accuracy = accuracy_score(y_test, y_pred)
sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
specificity = tn / (tn + fp) if (tn + fp) else 0.0
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("\nBest threshold:", round(best_t, 2))
print("Confusion Matrix:")
print("TN FP"); print(tn, fp)
print("FN TP"); print(fn, tp)

print(f"\nAccuracy     : {accuracy:.4f}")
print(f"Sensitivity  : {sensitivity:.4f}")
print(f"Specificity  : {specificity:.4f}")
print(f"F1 Score     : {f1:.4f}")
print(f"ROC-AUC      : {roc_auc:.4f}")
print(f"Balanced Acc : {best_ba:.4f}")

[I 2026-02-28 21:02:37,292] A new study created in memory with name: no-name-4bdba9c2-b313-423a-b650-79fc136aa528


Class weights: {0: 6.143518518518518, 1: 0.544298605414274}


[I 2026-02-28 21:02:56,405] Trial 0 finished with value: 0.6982392228293868 and parameters: {'batch_size': 32, 'n_layers': 1, 'units_0': 64, 'l2': 2.7592113317237197e-06, 'dropout': 0.23969651345215293, 'lr': 0.0005737385212327489, 'use_bn_0': False}. Best is trial 0 with value: 0.6982392228293868.
[I 2026-02-28 21:03:12,590] Trial 1 finished with value: 0.6946569520340011 and parameters: {'batch_size': 128, 'n_layers': 3, 'units_0': 256, 'l2': 8.928701287073253e-05, 'dropout': 0.45973895840904994, 'lr': 0.003063247058013408, 'use_bn_0': False, 'units_1': 128, 'use_bn_1': True, 'units_2': 16, 'use_bn_2': True}. Best is trial 0 with value: 0.6982392228293868.
[I 2026-02-28 21:04:14,489] Trial 2 finished with value: 0.7041287188828171 and parameters: {'batch_size': 128, 'n_layers': 3, 'units_0': 128, 'l2': 3.9077613598198475e-05, 'dropout': 0.1526226448052465, 'lr': 1.1662406339879326e-05, 'use_bn_0': False, 'units_1': 80, 'use_bn_1': False, 'units_2': 32, 'use_bn_2': False}. Best is tri


Best Balanced Accuracy: 0.7525197328476017
Best Params:
 {'batch_size': 64, 'n_layers': 1, 'units_0': 128, 'l2': 0.0001891353364270665, 'dropout': 0.28607412676429644, 'lr': 0.00023352562807626057, 'use_bn_0': True}
Epoch 1/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 5s 158ms/step - auc: 0.4619 - loss: 0.9841 - val_auc: 0.5180 - val_loss: 0.5252
Epoch 2/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - auc: 0.5681 - loss: 0.8260 - val_auc: 0.5440 - val_loss: 0.5407
Epoch 3/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - auc: 0.5690 - loss: 0.8081 - val_auc: 0.5692 - val_loss: 0.5552
Epoch 4/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - auc: 0.5899 - loss: 0.7779 - val_auc: 0.5897 - val_loss: 0.5670
Epoch 5/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - auc: 0.6924 - loss: 0.6908 - val_auc: 0.6086 - val_loss: 0.5767
Epoch 6/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - auc: 0.6526 - loss: 0.7206 - val_auc: 0.6293 - val_loss: 0.5864
Epoch 7/500
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - auc: 0.6595 - loss: 0.6971